In [ ]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv('../data/processed_cleaned_data.csv')

df['description'] = df['description'].fillna('Missing').astype(str)
df['title'] = df['title'].fillna('Missing').astype(str)
df['skills'] = df['skills'].fillna('Missing').astype(str)
df['company'] = df['company'].fillna('Missing').astype(str)
df['work_mode'] = df['work_mode'].fillna('Missing').astype(str)
df['duration'] = df['duration'].fillna('Missing').astype(str)
df['openings'] = df['openings'].fillna('0').astype(str)

df['has_website'] = df['company_website'].apply(lambda x: 0 if str(x) == 'Missing' else 1)
df['has_email'] = df['recruiter_email'].apply(lambda x: 0 if str(x) == 'Missing' else 1)

df['skills_count'] = df['skills'].apply(lambda x: 0 if str(x) == 'Missing' else len(str(x).split(',')))
df['title_word_count'] = df['title'].apply(lambda x: len(str(x).split()))
df['desc_char_length'] = df['description'].apply(lambda x: len(str(x)))

print(f"Base data environment successfully prepared. Rows: {df.shape[0]}")

Base data environment successfully prepared. Rows: 13616


In [ ]:
def clean_openings_count(val):
    digits = re.findall(r'\d+', val)
    return int(digits[0]) if digits else 1

df['numeric_openings'] = df['openings'].apply(clean_openings_count)

df['is_bulk_hiring'] = df['numeric_openings'].apply(lambda x: 1 if x >= 25 else 0)

df['is_wfh'] = df['work_mode'].str.lower().apply(lambda x: 1 if 'remote' in x or 'wfh' in x or 'home' in x else 0)
df['is_short_duration'] = df['duration'].str.lower().apply(lambda x: 1 if '1 month' in x or 'week' in x else 0)
df['is_wfh_short_duration'] = ((df['is_wfh'] == 1) & (df['is_short_duration'] == 1)).astype(int)

df['company_name_word_count'] = df['company'].apply(lambda x: len(str(x).split()))


url_pattern = r'https?://\S+|www\.\S+'
df['link_count_desc'] = df['description'].apply(lambda x: len(re.findall(url_pattern, x, flags=re.IGNORECASE)))

phone_pattern = r'\b\d{7,12}\b'
df['contact_count_desc'] = df['description'].apply(lambda x: len(re.findall(phone_pattern, x)))

def calculate_caps_ratio(text):
    letters = [char for char in text if char.isalpha()]
    return sum(1 for char in letters if char.isupper()) / len(letters) if letters else 0.0

df['all_caps_ratio_desc'] = df['description'].apply(calculate_caps_ratio)


scam_keywords = [
    'registration fee', 'security deposit', 'processing fee', 'training fee',
    'training charge', 'whatsapp directly', 'investment', 'earn online',
    'pay to work', 'income source', 'telegram link', 'typing job', 'form filling'
]
desc_lower_series = df['description'].str.lower()
df['keyword_count'] = desc_lower_series.apply(lambda text: sum(1 for word in scam_keywords if word in text))

df['has_urgency_words'] = desc_lower_series.str.contains('immediately|urgent|limited|hurry', regex=True).astype(int)
df['stipend_to_skills_ratio'] = df['cleaned_stipend_monthly'] / (df['skills_count'] + 1)
df['text_density_index'] = df['title_word_count'] / (df['desc_char_length'] + 1)
df['low_skills_high_urgency'] = ((df['skills_count'] <= 1) & (df['has_urgency_words'] == 1)).astype(int)
df['is_anonymous_recruiter'] = ((df['has_website'] == 0) & (df['has_email'] == 0)).astype(int)

print("All structural features across all dataset columns compiled perfectly with zero data leakage!")

All structural features across all dataset columns compiled perfectly with zero data leakage!


In [ ]:
ultimate_feature_lineup = [
    'is_scam',
    'cleaned_stipend_monthly',
    'has_website',
    'has_email',
    'skills_count',
    'title_word_count',
    'desc_char_length',
    'numeric_openings',
    'is_bulk_hiring',
    'is_wfh_short_duration',
    'company_name_word_count',
    'link_count_desc',
    'contact_count_desc',
    'all_caps_ratio_desc',
    'keyword_count',
    'stipend_to_skills_ratio',
    'text_density_index',
    'low_skills_high_urgency',
    'is_anonymous_recruiter'
]

# Generate the pure training slice matrix
model_ready_dataset = df[ultimate_feature_lineup].copy()

# Sweep and clean any remaining NaN entries safely with 0
model_ready_dataset = model_ready_dataset.fillna(0)

print("--- DEFINITIVE MODEL-READY GRID VERIFICATION ---")
print(f"Total Available Data Rows : {model_ready_dataset.shape[0]}")
print(f"Total Unique Model Features: {model_ready_dataset.shape[1]}")
print("\nFinal clean numeric feature lineup for Model Training:")
print(list(model_ready_dataset.columns))

--- DEFINITIVE MODEL-READY GRID VERIFICATION ---
Total Available Data Rows : 13616
Total Unique Model Features: 19

Final clean numeric feature lineup for Model Training:
['is_scam', 'cleaned_stipend_monthly', 'has_website', 'has_email', 'skills_count', 'title_word_count', 'desc_char_length', 'numeric_openings', 'is_bulk_hiring', 'is_wfh_short_duration', 'company_name_word_count', 'link_count_desc', 'contact_count_desc', 'all_caps_ratio_desc', 'keyword_count', 'stipend_to_skills_ratio', 'text_density_index', 'low_skills_high_urgency', 'is_anonymous_recruiter']


In [31]:
model_ready_dataset.to_csv('../data/model_ready_dataset.csv', index=False)
print("\nFinal model ready dataset saved to '../data/model_ready_dataset.csv'")


Final model ready dataset saved to '../data/model_ready_dataset.csv'
